In [0]:
%pip install elasticsearch==8.19.0
%restart_python

# Delete stale works from Elasticsearch (oxjob #784)

The works index sync (`sync_works`) only upserts, so a work removed from
`openalex.works.openalex_works` lived in `works-v34` forever — the same gap
behind the 2026-07-16 authors-index incident class. This notebook is the delete
path: it consumes the ledger `openalex.works.deleted_works` (written by
`notebooks/end2end/TrackDeletedWorks`) and bulk-deletes every entry not yet
stamped `es_deleted_at`.

- Runs after `Sync_to_Elasticsearch` in end2end. Deletes are 404-tolerant
  (already-absent docs count as done), so retries are safe.
- Rows are stamped `es_deleted_at` only after the delete pass; non-404 failures
  stay unstamped and are retried next night.
- **Guard**: aborts if `openalex_works` is empty or the pending set exceeds
  `guard_fraction` (default 0.5%) of the live index doc count, unless
  `guard_override=true` (job parameter `deleted_works_guard_override`).
- **`is_full_reconcile=true`** (manual, run-now-with-parameters): scrolls every
  `_id` out of the index and ledgers any doc whose work is not in
  `openalex_works` — catches historical ghosts that predate the ledger. Slow
  (full index scroll); intended as a one-off backfill after this ships.


In [0]:
import logging
from pyspark.sql import functions as F
from elasticsearch import Elasticsearch, helpers

logging.basicConfig(level=logging.WARNING, format="[%(asctime)s]: %(message)s")
log = logging.getLogger(__name__)

ELASTIC_INDEX = "works-v34"
ELASTIC_URL = dbutils.secrets.get(scope="elastic", key="elastic_url")
ID_PREFIX = "https://openalex.org/W"

dbutils.widgets.text("guard_override", "false")
dbutils.widgets.text("guard_fraction", "0.005")
dbutils.widgets.text("is_full_reconcile", "false")
dbutils.widgets.text("env_suffix", "")

GUARD_OVERRIDE = dbutils.widgets.get("guard_override").lower() == "true"
GUARD_FRACTION = float(dbutils.widgets.get("guard_fraction"))
IS_FULL_RECONCILE = dbutils.widgets.get("is_full_reconcile").lower() == "true"
ENV_SUFFIX = dbutils.widgets.get("env_suffix")

CATALOG = f"openalex{ENV_SUFFIX}"
WORKS = f"{CATALOG}.works.openalex_works"
LEDGER = f"{CATALOG}.works.deleted_works"
ES_SCAN_SCRATCH = f"{CATALOG}.works._tmp_es_work_ids_784"

client = Elasticsearch(
    hosts=[ELASTIC_URL],
    request_timeout=180,
    max_retries=5,
    retry_on_timeout=True,
)

print(f"guard_override: {GUARD_OVERRIDE}, guard_fraction: {GUARD_FRACTION}, full_reconcile: {IS_FULL_RECONCILE}")


### Optional: full ES-scan reconcile (manual backfill of historical ghosts)

In [0]:
if IS_FULL_RECONCILE:
    print(f"FULL RECONCILE: scrolling all _ids from {ELASTIC_INDEX} into {ES_SCAN_SCRATCH}...", flush=True)
    spark.sql(f"DROP TABLE IF EXISTS {ES_SCAN_SCRATCH}")
    spark.sql(f"CREATE TABLE {ES_SCAN_SCRATCH} (id STRING) USING DELTA")

    batch, total = [], 0
    def flush(rows):
        if rows:
            spark.createDataFrame([(r,) for r in rows], "id STRING") \
                .write.mode("append").saveAsTable(ES_SCAN_SCRATCH)

    for hit in helpers.scan(
        client, index=ELASTIC_INDEX, query={"query": {"match_all": {}}},
        _source=False, size=10_000, scroll="15m",
    ):
        batch.append(hit["_id"])
        if len(batch) >= 1_000_000:
            flush(batch)
            total += len(batch)
            batch = []
            print(f"  ...scanned {total:,}", flush=True)
    flush(batch)
    total += len(batch)
    print(f"Scanned {total:,} ES ids.", flush=True)

    unparseable = spark.sql(f"""
        SELECT COUNT(*) AS cnt FROM {ES_SCAN_SCRATCH}
        WHERE TRY_CAST(REGEXP_EXTRACT(id, 'W([0-9]+)$', 1) AS BIGINT) IS NULL
    """).collect()[0].cnt
    if unparseable:
        print(f"WARNING: {unparseable:,} ES _ids do not match {ID_PREFIX}<digits>; skipped.")

    ghosts = spark.sql(f"""
        INSERT INTO {LEDGER}
        SELECT e.work_id, current_date(), current_timestamp(), NULL
        FROM (
            SELECT TRY_CAST(REGEXP_EXTRACT(id, 'W([0-9]+)$', 1) AS BIGINT) AS work_id
            FROM {ES_SCAN_SCRATCH}
        ) e
        LEFT ANTI JOIN {WORKS} w ON e.work_id = w.id
        LEFT ANTI JOIN {LEDGER} l ON e.work_id = l.work_id
        WHERE e.work_id IS NOT NULL
    """).collect()[0].num_inserted_rows
    print(f"Full reconcile ledgered {ghosts:,} historical ghost docs.")
else:
    print("Incremental run (ledger-driven only).")


### Delete pass

In [0]:
pending_df = spark.sql(f"SELECT work_id FROM {LEDGER} WHERE es_deleted_at IS NULL")
pending_count = pending_df.count()
works_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {WORKS}").collect()[0].cnt
live_count = client.count(index=ELASTIC_INDEX)["count"]
print(f"Pending deletes: {pending_count:,}; live index docs: {live_count:,}; {WORKS}: {works_count:,}")

if pending_count > 0:
    if works_count == 0:
        raise Exception(f"ABORT: {WORKS} has 0 rows; refusing to trust the ledger. No deletes issued.")
    if pending_count > GUARD_FRACTION * live_count and not GUARD_OVERRIDE:
        raise Exception(
            f"ABORT: {pending_count:,} pending deletes (> {GUARD_FRACTION:.2%} of {live_count:,} live docs). "
            "No deletes issued; ledger unchanged. If this is a sanctioned mass deletion, "
            "re-run with deleted_works_guard_override=true."
        )

    ids = [r.work_id for r in pending_df.collect()]

    def actions():
        for wid in ids:
            yield {"_op_type": "delete", "_index": ELASTIC_INDEX, "_id": f"{ID_PREFIX}{wid}"}

    deleted = missing = 0
    failed_ids = []
    for success, info in helpers.parallel_bulk(
        client, actions(), chunk_size=700, thread_count=4,
        raise_on_error=False, raise_on_exception=False,
    ):
        item = info.get("delete", {})
        if success:
            deleted += 1
            if deleted % 100_000 == 0:
                print(f"  ...deleted {deleted:,} / {len(ids):,}", flush=True)
        elif item.get("status") == 404:
            missing += 1
        else:
            failed_ids.append(int(str(item.get("_id", "0")).rsplit("W", 1)[-1] or 0))
            if len(failed_ids) <= 10:
                print(f"  DELETE failed: {info}", flush=True)

    print(f"Deleted {deleted:,}, already absent {missing:,}, failed {len(failed_ids):,} of {len(ids):,}.")

    if failed_ids:
        spark.createDataFrame([(i,) for i in failed_ids], "work_id BIGINT") \
            .createOrReplaceTempView("failed_deletes")
        spark.sql(f"""
            UPDATE {LEDGER} SET es_deleted_at = current_timestamp()
            WHERE es_deleted_at IS NULL
              AND work_id NOT IN (SELECT work_id FROM failed_deletes)
        """)
        print(f"Stamped all but {len(failed_ids):,} failures (left pending for retry).")
    else:
        spark.sql(f"UPDATE {LEDGER} SET es_deleted_at = current_timestamp() WHERE es_deleted_at IS NULL")
        print("Stamped es_deleted_at on all pending rows.")
else:
    print("Nothing to delete.")

spark.sql(f"DROP TABLE IF EXISTS {ES_SCAN_SCRATCH}")
client.indices.refresh(index=ELASTIC_INDEX)
print(f"{client.count(index=ELASTIC_INDEX)['count']:,} documents remain in {ELASTIC_INDEX}.")
client.close()
